# Módulo 3 — Análisis t-SNE de conversaciones (bonus)

**Requisitos:**
- Tabla `conversation_logs` (migración `002`)
- Datos: `python scripts/seed_conversation_logs.py --api http://127.0.0.1:8000`
- `.env` con `OPENAI_API_KEY`, `SUPABASE_URL`, `SUPABASE_KEY`

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from sklearn.manifold import TSNE
from supabase import create_client

load_dotenv(Path('..') / '.env')

client = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
rows = client.table('conversation_logs').select(
    'session_id, transcript, user_message, channel'
).order('created_at', desc=True).limit(500).execute().data
print(f'Conversaciones cargadas: {len(rows)}')
assert len(rows) >= 5, 'Ejecuta seed_conversation_logs.py antes del análisis'

In [ ]:
def _label(msg: str) -> str:
    m = (msg or '').lower()
    if 'nit' in m or 'legal' in m or 'razón' in m:
        return 'legal/NIT'
    if 'contacto' in m or 'teléfono' in m or 'correo' in m or 'whatsapp' in m:
        return 'contacto'
    if 'junta' in m or 'directiva' in m:
        return 'gobierno'
    if 'sostenib' in m or 'ambient' in m or 'certific' in m:
        return 'sostenibilidad'
    if 'línea' in m or 'negocio' in m or 'caña' in m:
        return 'negocio'
    return 'otros'

texts = [r.get('transcript') or r.get('user_message', '') for r in rows]
labels = [_label(r.get('user_message', '')) for r in rows]
embeddings = OpenAIEmbeddings(model=os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small'))
vectors = np.array(embeddings.embed_documents(texts))
print('Shape:', vectors.shape)

In [ ]:
perplexity = min(30, max(5, len(texts) // 3))
xy = TSNE(n_components=2, perplexity=perplexity, random_state=42).fit_transform(vectors)

palette = {
    'legal/NIT': '#1b5e20',
    'contacto': '#2e7d32',
    'gobierno': '#f57c00',
    'sostenibilidad': '#0288d1',
    'negocio': '#6a1b9a',
    'otros': '#757575',
}

plt.figure(figsize=(11, 8))
for lab in sorted(set(labels)):
    mask = np.array(labels) == lab
    plt.scatter(xy[mask, 0], xy[mask, 1], label=lab, alpha=0.75, c=palette.get(lab, '#333'))
plt.legend(title='Intención (heurística)')
plt.title('t-SNE — conversaciones del asistente Riopaila')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
out = Path('..') / 'docs' / 'tsne_conversaciones.png'
plt.savefig(out, dpi=150)
plt.show()
print('Guardado:', out)

## Interpretación (para el informe final)

- **Clústeres cercanos del mismo color:** intenciones similares (p. ej. varias preguntas de NIT/legal).
- **contacto vs legal:** suelen separarse si los embeddings capturan vocabulario distinto.
- **Puntos aislados (otros):** consultas atípicas, errores de RAG o mensajes muy cortos.
- **Uso operativo:** detectar si muchas conversaciones caen en "otros" → ampliar `company_info` o mejorar chunking.

Incluye el PNG `docs/tsne_conversaciones.png` en el informe PDF del Módulo 3.